# Notebook 04: Deep Sequential Time-Series Forecasting (LSTM) for Crop Yield

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fahadrahman444/scientific-research-and-methodologies/blob/main/colab_notebooks/04_Deep_Learning_TimeSeries_Forecasting_Colab.ipynb)

**Research Topic:** A Data-Driven Analysis of 15-Year Weather Variability and Its Impact on Agricultural Crop Yield in Bangladesh  
**Model Architecture:** PyTorch LSTM Recurrent Neural Network  
**Hardware Target:** Google Colab GPU (T4 / V100)  

---
## 1. Deep Learning Pipeline Overview
This notebook designs a multi-step sequence forecasting model using PyTorch LSTM to project Bangladesh rice yield based on historical meteorological features.


In [ ]:
# Step 1: GPU Verification and Library Setup
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device in use: {device}')
if torch.cuda.is_available():
    print(f'GPU Device Name: {torch.cuda.get_device_name(0)}')


In [ ]:
# Step 2: Load Integrated FAOSTAT & Climate Dataset
INTEGRATED_LOCAL = '../data/processed/bangladesh_faostat_weather_integrated_15yr.csv'
INTEGRATED_URL = f'{BASE_RAW_GITHUB}/data/processed/bangladesh_faostat_weather_integrated_15yr.csv'

df = pd.read_csv(INTEGRATED_LOCAL) if os.path.exists(INTEGRATED_LOCAL) else pd.read_csv(INTEGRATED_URL)
features = ['tmax', 'tmin', 'rain', 'spi', 'heat_days', 'humidity', 'yield_kg_ha']
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df[features])
print(f'Normalized feature array of shape {scaled_data.shape}.')


In [ ]:
# Step 3: Sequence Creation (Rolling 3-Year Time Horizon)
def make_sequences(data, seq_len=3):
    xs, ys = [], []
    for i in range(len(data) - seq_len):
        x = data[i:(i + seq_len), :-1] # weather features
        y = data[i + seq_len, -1]       # yield
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

X_seq, y_seq = make_sequences(scaled_data, seq_len=3)
print(f'Created {len(X_seq)} sequential samples.')


In [ ]:
# Step 4: PyTorch LSTM Network Definition
class YieldLSTM(nn.Module):
    def __init__(self, in_features, hidden_size=32, num_layers=2):
        super(YieldLSTM, self).__init__()
        self.lstm = nn.LSTM(in_features, hidden_size, num_layers=num_layers, batch_first=True, dropout=0.1)
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        return self.fc(last_hidden).squeeze()

model = YieldLSTM(in_features=X_seq.shape[2]).to(device)
print(model)


In [ ]:
# Step 5: Model Training Loop
X_tensor = torch.tensor(X_seq, dtype=torch.float32).to(device)
y_tensor = torch.tensor(y_seq, dtype=torch.float32).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

epochs = 150
losses = []
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    pred = model(X_tensor)
    loss = criterion(pred, y_tensor)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 30 == 0:
        print(f'Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.5f}')

plt.figure(figsize=(9, 4))
plt.plot(losses, color='#1f77b4', lw=2)
plt.title('PyTorch LSTM Training Convergence Curve', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.tight_layout()
plt.show()
